In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import joblib
import torch
from sklearn.metrics import f1_score

import config, models, attacks, realism

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Load CICIoV processed arrays, scaler, encoder
arrays = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
Xc_train, yc_train = arrays["X_train"], arrays["y_train"]
Xc_test,  yc_test  = arrays["X_test"],  arrays["y_test"]
cic_scaler  = joblib.load(config.PROCESSED_DIR / "feature_scaler.joblib")
cic_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
cic_class_names = list(cic_encoder.classes_)

print("CICIoV X_test:", Xc_test.shape, " classes:", cic_class_names)

# Train the CICIoV baseline CNN and wrap for ART
cic_cnn = models.CNN1D(n_features=Xc_train.shape[1], n_classes=len(cic_class_names))
cic_cnn = models.train_cnn(cic_cnn, Xc_train, yc_train, n_epochs=50,
                           device=DEVICE, random_seed=config.RANDOM_SEED)
cic_clf = attacks.wrap_cnn_for_art(cic_cnn, n_features=Xc_train.shape[1],
                                   n_classes=len(cic_class_names), device=DEVICE)

# --- Build the two benign envelopes ---
# 1) Dedup benign: benign rows from strict train+test (parallels ROAD)
cic_train = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_train_dup.csv")
cic_test  = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_test.csv")
benign_dedup = pd.concat([
    cic_train[cic_train["true_class"] == "benign"],
    cic_test[cic_test["true_class"] == "benign"],
], ignore_index=True)

# 2) Raw benign: the full original benign file (widest envelope)
raw_benign = pd.read_csv(config.RAW_DIR / config.RAW_FILES["benign"])
raw_benign.columns = [c.strip() for c in raw_benign.columns]

print(f"benign (dedup): {len(benign_dedup):,} frames")
print(f"benign (raw)  : {len(raw_benign):,} frames")

ranges_dedup = realism.learn_observed_ranges(benign_dedup, config.ID_COLUMN, config.DATA_COLUMNS)
ranges_raw   = realism.learn_observed_ranges(raw_benign,   config.ID_COLUMN, config.DATA_COLUMNS)
print(f"dedup benign IDs: {len(ranges_dedup)}   raw benign IDs: {len(ranges_raw)}")

id_idx = config.FEATURE_COLUMNS.index(config.ID_COLUMN)
data_idx = [config.FEATURE_COLUMNS.index(c) for c in config.DATA_COLUMNS]

Device: cuda
CICIoV X_test: (718, 9)  classes: ['DoS', 'benign', 'spoofing-GAS', 'spoofing-RPM', 'spoofing-SPEED', 'spoofing-STEERING_WHEEL']
    epoch   1/50     loss 1.0606
    epoch   5/50     loss 0.2044
    epoch  10/50     loss 0.0197
    epoch  15/50     loss 0.0067
    epoch  20/50     loss 0.0043
    epoch  25/50     loss 0.0034
    epoch  30/50     loss 0.0025
    epoch  35/50     loss 0.0018
    epoch  40/50     loss 0.0018
    epoch  45/50     loss 0.0014
    epoch  50/50     loss 0.0014
benign (dedup): 3,547 frames
benign (raw)  : 1,223,737 frames
dedup benign IDs: 72   raw benign IDs: 72


In [2]:
Xc_test_f = Xc_test.astype(np.float32)

def macro_f1c(y_pred):
    return f1_score(yc_test, y_pred, average="macro", zero_division=0)

# Helper: split rejection into "unknown ID" vs "byte out of range"
def rejection_breakdown(X_int, ranges):
    n = len(X_int)
    unknown_id = np.zeros(n, dtype=bool)
    out_of_range = np.zeros(n, dtype=bool)
    for i in range(n):
        idv = int(round(X_int[i, id_idx]))
        if idv not in ranges:
            unknown_id[i] = True
            continue
        idr = ranges[idv]
        bad = False
        for col_name, col_idx in zip(idr.keys(), data_idx):
            lo, hi = idr[col_name]
            if X_int[i, col_idx] < lo or X_int[i, col_idx] > hi:
                bad = True
                break
        out_of_range[i] = bad
    return unknown_id, out_of_range

print(f"{'eps':>6}  {'raw_adv':>8}  {'rounded':>8}  {'rej_dedup':>10}  {'rej_raw':>8}  {'unkID%':>7}  {'range%':>7}")

cic_threat_rows = []
for eps in config.FGSM_EPSILONS:
    X_adv = attacks.generate_pgd(cic_clf, Xc_test_f, epsilon=eps)

    f1_raw = macro_f1c(cic_clf.predict(X_adv).argmax(axis=1))
    X_round_scaled, X_int = realism.round_to_integer_frames(
        X_adv, cic_scaler, config.FEATURE_MIN, config.FEATURE_MAX,
    )
    f1_round = macro_f1c(cic_clf.predict(X_round_scaled).argmax(axis=1))

    # Rejection under each envelope
    mask_dedup = realism.observed_range_mask(X_int, ranges_dedup, id_idx, data_idx)
    mask_raw   = realism.observed_range_mask(X_int, ranges_raw,   id_idx, data_idx)
    rej_dedup = 100.0 * (~mask_dedup).sum() / len(mask_dedup)
    rej_raw   = 100.0 * (~mask_raw).sum()   / len(mask_raw)

    # Breakdown (using raw envelope): unknown ID vs byte out of range
    unk, oor = rejection_breakdown(X_int, ranges_raw)
    pct_unk = 100.0 * unk.sum() / len(unk)
    pct_oor = 100.0 * oor.sum() / len(oor)

    cic_threat_rows.append({
        "eps": eps, "f1_raw": f1_raw, "f1_round": f1_round,
        "rej_dedup": rej_dedup, "rej_raw": rej_raw,
        "pct_unknown_id": pct_unk, "pct_out_of_range": pct_oor,
    })
    print(f"{eps:>6.2f}  {f1_raw:>8.3f}  {f1_round:>8.3f}  {rej_dedup:>9.1f}%  {rej_raw:>7.1f}%  {pct_unk:>6.1f}%  {pct_oor:>6.1f}%")

print(f"\n(CICIoV test set: {len(yc_test)} frames)")

   eps   raw_adv   rounded   rej_dedup   rej_raw   unkID%   range%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01     0.703     0.467      100.0%    100.0%   100.0%     0.0%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05     0.450     0.356      100.0%    100.0%    99.7%     0.3%


...

  0.10     0.199     0.133      100.0%    100.0%    99.9%     0.1%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.20     0.118     0.040      100.0%    100.0%    99.9%     0.1%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.30     0.095     0.046      100.0%    100.0%    99.6%     0.4%

(CICIoV test set: 718 frames)


## ID-constrained re-run: perturb payload bytes only

The earlier CICIoV threat-sizing was confounded because perturbing the continuous
ID feature knocked frames onto unknown IDs, dominating rejection. Here the ID is
frozen (mask=0) so only the 8 payload bytes are perturbed. This isolates whether
the CICIoV payload sits closer to benign than ROAD's, the real mechanism test.

In [4]:
import numpy as np

# Build the ID-freeze mask: 0 at ID index, 1 at the 8 payload byte indices.
# FEATURE_COLUMNS = [ID, DATA_0..DATA_7], so index 0 is ID.
mask = np.ones(len(config.FEATURE_COLUMNS), dtype=np.float32)
mask[id_idx] = 0.0
print("mask (0=frozen, 1=perturbable):", mask)

Xc_test_f = Xc_test.astype(np.float32)

def macro_f1c(y_pred):
    return f1_score(yc_test, y_pred, average="macro", zero_division=0)

print(f"\n{'eps':>6}  {'raw_adv':>8}  {'rounded':>8}  {'rej_raw':>8}  {'unkID%':>7}  {'range%':>7}")

cic_masked_rows = []
for eps in config.FGSM_EPSILONS:
    # PGD with the ID frozen: only payload bytes move
    X_adv = attacks.generate_pgd(cic_clf, Xc_test_f, epsilon=eps, mask=mask)

    f1_raw = macro_f1c(cic_clf.predict(X_adv).argmax(axis=1))
    X_round_scaled, X_int = realism.round_to_integer_frames(
        X_adv, cic_scaler, config.FEATURE_MIN, config.FEATURE_MAX,
    )
    f1_round = macro_f1c(cic_clf.predict(X_round_scaled).argmax(axis=1))

    mask_ok = realism.observed_range_mask(X_int, ranges_raw, id_idx, data_idx)
    rej_raw = 100.0 * (~mask_ok).sum() / len(mask_ok)

    unk, oor = rejection_breakdown(X_int, ranges_raw)
    pct_unk = 100.0 * unk.sum() / len(unk)
    pct_oor = 100.0 * oor.sum() / len(oor)

    cic_masked_rows.append({
        "eps": eps, "f1_raw": f1_raw, "f1_round": f1_round,
        "rej_raw": rej_raw, "pct_unknown_id": pct_unk, "pct_out_of_range": pct_oor,
    })
    print(f"{eps:>6.2f}  {f1_raw:>8.3f}  {f1_round:>8.3f}  {rej_raw:>7.1f}%  {pct_unk:>6.1f}%  {pct_oor:>6.1f}%")

print(f"\n(CICIoV test set: {len(yc_test)} frames, ID frozen)")

mask (0=frozen, 1=perturbable): [0. 1. 1. 1. 1. 1. 1. 1. 1.]

   eps   raw_adv   rounded   rej_raw   unkID%   range%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01     0.703     0.467    100.0%    75.9%    24.1%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05     0.577     0.370    100.0%    75.9%    24.1%


...

  0.10     0.310     0.083    100.0%    75.9%    24.1%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.20     0.141     0.030    100.0%    75.9%    24.1%


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.30     0.109     0.056    100.0%    75.9%    24.1%

(CICIoV test set: 718 frames, ID frozen)


In [5]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# CICIoV benign reference in scaled space (train benign, parallels ROAD section 48)
cic_train = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_train_dup.csv")
cic_benign = cic_train[cic_train["true_class"] == "benign"]
benign_scaled = cic_scaler.transform(cic_benign[config.FEATURE_COLUMNS].values).astype(np.float32)

nn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(benign_scaled)

# Per attack class: mean L2 distance to nearest benign, in scaled space
cic_test_df = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_test.csv")
print(f"{'class':26s} {'test_sigs':>9} {'mean_dist':>10} {'median_dist':>12}")
for name in ["DoS", "spoofing-GAS", "spoofing-RPM", "spoofing-SPEED", "spoofing-STEERING_WHEEL"]:
    atk = cic_test_df[cic_test_df["true_class"] == name]
    if len(atk) == 0:
        continue
    atk_scaled = cic_scaler.transform(atk[config.FEATURE_COLUMNS].values).astype(np.float32)
    d, _ = nn.kneighbors(atk_scaled)
    d = d.ravel()
    print(f"{name:26s} {len(atk):>9} {d.mean():>10.4f} {np.median(d):>12.4f}")

print("\nROAD attack distances for comparison (section 48):")
print("  fuzzing 0.500, max-speedometer 0.423, reverse-light-on 0.086, reverse-light-off 0.044")

class                      test_sigs  mean_dist  median_dist
DoS                                4     0.2320       0.2320
spoofing-GAS                       1     0.6661       0.6661
spoofing-RPM                       2     0.6219       0.6219
spoofing-SPEED                     1     0.2090       0.2090
spoofing-STEERING_WHEEL            1     0.3082       0.3082

ROAD attack distances for comparison (section 48):
  fuzzing 0.500, max-speedometer 0.423, reverse-light-on 0.086, reverse-light-off 0.044
